# Silver: Quarantine Routing

Manages the quarantine table for failed data quality records.

## Quarantine Reasons

1. **DQ_FAILURE**: Data quality validation failed
2. **PARSE_ERROR**: Unable to parse required fields
3. **REFERENTIAL_INTEGRITY**: Foreign key violations
4. **BUSINESS_RULE_VIOLATION**: Business logic checks failed
5. **DUPLICATE_DETECTED**: High-confidence duplicate

## Actions

- **List**: View quarantined records
- **Reinstate**: Move records back to active (after correction)
- **Purge**: Permanently remove from quarantine
- **Export**: Export quarantine for external review

## Integration

Quarantined records are excluded from downstream processing until reinstated.

In [0]:
dbutils.widgets.dropdown("action", "list", ["list", "reinstate", "purge", "summary"], "Action")
dbutils.widgets.text("enterprise_job_ids", "", "Comma-separated job IDs for reinstate/purge")
dbutils.widgets.text("reason_filter", "", "Filter by quarantine reason")
dbutils.widgets.text("limit", "100", "Result limit for list")

action = dbutils.widgets.get("action")
enterprise_job_ids = dbutils.widgets.get("enterprise_job_ids").strip()
reason_filter = dbutils.widgets.get("reason_filter").strip()
limit = int(dbutils.widgets.get("limit"))

print(f"Action: {action}")
print(f"Job IDs: {enterprise_job_ids if enterprise_job_ids else 'N/A'}")
print(f"Reason Filter: {reason_filter if reason_filter else 'All'}")
print(f"Limit: {limit}")

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
QUARANTINE_SCHEMA = f"{CATALOG}.quarantine"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {run_id}")

In [0]:
if action == "list":
    query = f"""
    SELECT 
        q.quarantine_id,
        q.enterprise_job_id,
        q.source_name,
        q.source_job_id,
        q.failure_stage,
        q.failed_rule_name,
        q.severity,
        q.quarantined_at,
        q.reprocess_status,
        j.company_name_raw,
        j.title_raw
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs q
    LEFT JOIN {SILVER_SCHEMA}.silver_jobs_current j
        ON q.enterprise_job_id = j.enterprise_job_id
    WHERE 1=1
    """
    
    if reason_filter:
        query += f" AND q.failed_rule_name LIKE '%{reason_filter}%'"
    
    query += f" ORDER BY q.quarantined_at DESC LIMIT {limit}"
    
    quarantine_df = spark.sql(query)
    quarantine_count = quarantine_df.count()
    
    print(f"\n=== Quarantined Records: {quarantine_count} ===")
    
    if quarantine_count > 0:
        display(quarantine_df)
    else:
        print("No quarantined records found")
    
    result = {
        "status": "success",
        "action": "list",
        "quarantine_count": quarantine_count
    }
    
    dbutils.notebook.exit(json.dumps(result))

In [0]:
if action == "reinstate":
    if not enterprise_job_ids:
        dbutils.notebook.exit({"status": "error", "message": "enterprise_job_ids required for reinstate"})
    
    job_id_list = [f"'{jid.strip()}'" for jid in enterprise_job_ids.split(",")]
    
    # Get quarantined records
    quarantine_query = f"""
    SELECT enterprise_job_id, source_name
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs
    WHERE enterprise_job_id IN ({','.join(job_id_list)})
    """
    
    to_reinstate_df = spark.sql(quarantine_query)
    reinstate_count = to_reinstate_df.count()
    
    if reinstate_count == 0:
        dbutils.notebook.exit({"status": "error", "message": "No matching quarantined records found"})
    
    # Update quarantine status to RESOLVED
    spark.sql(f"""
    UPDATE {QUARANTINE_SCHEMA}.quarantine_jobs
    SET reprocess_status = 'RESOLVED',
        resolved_at = current_timestamp(),
        reprocess_batch_id = '{run_id}'
    WHERE enterprise_job_id IN ({','.join(job_id_list)})
    """)
    
    # Restore in current table (clear soft_delete_flag if set)
    delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
    
    delta_current.update(
        condition=F.col("enterprise_job_id").isin([jid.strip() for jid in enterprise_job_ids.split(",")]),
        set={
            "soft_delete_flag": F.lit(False),
            "soft_delete_reason": F.lit(None),
            "is_active": F.lit(True),
            "updated_at": run_timestamp
        }
    )
    
    print(f"Reinstated {reinstate_count} records from quarantine")
    
    # Log change
    spark.sql(f"""
    INSERT INTO {SILVER_SCHEMA}.silver_job_changes
    SELECT 
        uuid() as change_id,
        enterprise_job_id,
        source_name,
        'RESTORE' as change_type,
        CAST(NULL AS ARRAY<STRING>) as changed_columns,
        NULL as old_value_json,
        NULL as new_value_json,
        current_timestamp() as change_timestamp,
        '{run_id}' as batch_id
    FROM ({quarantine_query})
    """)
    
    result = {
        "status": "success",
        "action": "reinstate",
        "reinstated_count": reinstate_count
    }
    
    dbutils.notebook.exit(json.dumps(result))

In [0]:
if action == "purge":
    if not enterprise_job_ids:
        dbutils.notebook.exit({"status": "error", "message": "enterprise_job_ids required for purge"})
    
    job_id_list = [f"'{jid.strip()}'" for jid in enterprise_job_ids.split(",")]
    
    # Count records to purge
    purge_count = spark.sql(f"""
    SELECT COUNT(*) as cnt
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs
    WHERE enterprise_job_id IN ({','.join(job_id_list)})
    """).first()["cnt"]
    
    if purge_count == 0:
        dbutils.notebook.exit({"status": "error", "message": "No matching quarantined records found"})
    
    # Mark as DISCARDED (don't delete, preserve audit trail)
    spark.sql(f"""
    UPDATE {QUARANTINE_SCHEMA}.quarantine_jobs
    SET reprocess_status = 'DISCARDED',
        resolved_at = current_timestamp(),
        reprocess_batch_id = '{run_id}'
    WHERE enterprise_job_id IN ({','.join(job_id_list)})
    """)
    
    print(f"Marked {purge_count} records as DISCARDED in quarantine")
    print("⚠️ Records remain soft-deleted in current table")
    
    result = {
        "status": "success",
        "action": "purge",
        "purged_count": purge_count
    }
    
    dbutils.notebook.exit(json.dumps(result))

In [0]:
if action == "summary":
    # Overall quarantine stats
    overall_stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_quarantined,
        COUNT(DISTINCT source_name) as sources_affected,
        MIN(quarantined_at) as oldest_quarantine,
        MAX(quarantined_at) as newest_quarantine,
        SUM(CASE WHEN reprocess_status = 'PENDING' THEN 1 ELSE 0 END) as pending_count,
        SUM(CASE WHEN reprocess_status = 'RESOLVED' THEN 1 ELSE 0 END) as resolved_count,
        SUM(CASE WHEN reprocess_status = 'DISCARDED' THEN 1 ELSE 0 END) as discarded_count
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs
    """)
    
    print("\n=== Quarantine Overview ===")
    display(overall_stats)
    
    # Failure stage breakdown
    stage_breakdown = spark.sql(f"""
    SELECT 
        failure_stage,
        COUNT(*) as count,
        COUNT(DISTINCT source_name) as sources
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs
    GROUP BY failure_stage
    ORDER BY count DESC
    """)
    
    print("\n=== Failure Stage Breakdown ===")
    display(stage_breakdown)
    
    # Failed rule breakdown
    rule_breakdown = spark.sql(f"""
    SELECT 
        failed_rule_name,
        severity,
        COUNT(*) as count,
        COUNT(DISTINCT source_name) as sources
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs
    GROUP BY failed_rule_name, severity
    ORDER BY count DESC
    """)
    
    print("\n=== Failed Rules ===")
    display(rule_breakdown)
    
    # Source breakdown
    source_breakdown = spark.sql(f"""
    SELECT 
        source_name,
        COUNT(*) as quarantine_count,
        MIN(quarantined_at) as first_quarantine,
        MAX(quarantined_at) as last_quarantine,
        SUM(CASE WHEN reprocess_status = 'PENDING' THEN 1 ELSE 0 END) as pending
    FROM {QUARANTINE_SCHEMA}.quarantine_jobs
    GROUP BY source_name
    ORDER BY quarantine_count DESC
    """)
    
    print("\n=== Source Breakdown ===")
    display(source_breakdown)
    
    result = {
        "status": "success",
        "action": "summary"
    }
    
    dbutils.notebook.exit(json.dumps(result))